In [0]:
%sql
CREATE OR REPLACE TABLE silver.customers AS
SELECT DISTINCT *
FROM bronze.customers;

In [0]:
%sql
DESCRIBE bronze.orders;

In [0]:
%sql
CREATE OR REPLACE TABLE silver.orders AS
SELECT
    order_id,
    customer_id,
    restaurant_id,
    agent_id,
    delivery_location_id,
    order_status,
    payment_method,
    CAST(timestamp AS TIMESTAMP) AS order_timestamp,
    tip,
    total_amount
FROM bronze.orders;

In [0]:
%sql
CREATE OR REPLACE TABLE silver.order_items AS
SELECT
    order_id,
    item_id,
    item_name,
    CAST(item_price AS DECIMAL(10,2)) AS item_price,
    CAST(quantity AS INT) AS quantity
FROM bronze.order_items;

In [0]:
%sql
CREATE OR REPLACE TABLE silver.restaurants AS
SELECT * FROM bronze.restaurants;

CREATE OR REPLACE TABLE silver.delivery_agents AS
SELECT * FROM bronze.delivery_agents;

CREATE OR REPLACE TABLE silver.locations AS
SELECT * FROM bronze.locations;

CREATE OR REPLACE TABLE silver.menu_items AS
SELECT * FROM bronze.menu_items;

In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS duplicate_keys
FROM (
    SELECT customer_id
    FROM bronze.customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'restaurants', COUNT(*)
FROM (
    SELECT restaurant_id
    FROM bronze.restaurants
    GROUP BY restaurant_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'delivery_agents', COUNT(*)
FROM (
    SELECT agent_id
    FROM bronze.delivery_agents
    GROUP BY agent_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'locations', COUNT(*)
FROM (
    SELECT location_id
    FROM bronze.locations
    GROUP BY location_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'menu_items', COUNT(*)
FROM (
    SELECT item_id
    FROM bronze.menu_items
    GROUP BY item_id
    HAVING COUNT(*) > 1
);

In [0]:
%sql
SELECT *
FROM bronze.delivery_agents
WHERE agent_id IN (
    SELECT agent_id
    FROM bronze.delivery_agents
    GROUP BY agent_id
    HAVING COUNT(*) > 1
)
ORDER BY agent_id;

In [0]:
%sql
SELECT
    COUNT(*) AS duplicate_agent_keys,
    SUM(cnt - 1) AS extra_duplicate_rows
FROM (
    SELECT agent_id, COUNT(*) AS cnt
    FROM bronze.delivery_agents
    GROUP BY agent_id
    HAVING COUNT(*) > 1
);

In [0]:
%sql
SELECT
    COUNT(*) AS duplicate_groups_with_different_data
FROM (
    SELECT
        agent_id
    FROM bronze.delivery_agents
    GROUP BY agent_id
    HAVING COUNT(DISTINCT CONCAT_WS(
        '||',
        COALESCE(name, ''),
        COALESCE(phone, ''),
        CAST(COALESCE(rating, 0) AS STRING)
    )) > 1
);

In [0]:
%sql
-- Clean customers
CREATE OR REPLACE TABLE silver.customers AS
SELECT DISTINCT *
FROM bronze.customers;


-- Clean delivery agents
CREATE OR REPLACE TABLE silver.delivery_agents AS
SELECT DISTINCT *
FROM bronze.delivery_agents;

In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS duplicate_keys
FROM (
    SELECT customer_id
    FROM silver.customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'delivery_agents', COUNT(*)
FROM (
    SELECT agent_id
    FROM silver.delivery_agents
    GROUP BY agent_id
    HAVING COUNT(*) > 1
);

In [0]:
%sql
SELECT 'orders' AS check_name, COUNT(*) AS bad_rows
FROM (
    SELECT order_id
    FROM bronze.orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'order_items_duplicate_keys', COUNT(*)
FROM (
    SELECT order_id, item_id
    FROM bronze.order_items
    GROUP BY order_id, item_id
    HAVING COUNT(*) > 1
)

UNION ALL

SELECT 'orders_null_order_id', COUNT(*)
FROM bronze.orders
WHERE order_id IS NULL

UNION ALL

SELECT 'order_items_null_order_id', COUNT(*)
FROM bronze.order_items
WHERE order_id IS NULL

UNION ALL

SELECT 'order_items_null_item_id', COUNT(*)
FROM bronze.order_items
WHERE item_id IS NULL;

In [0]:
%sql
SELECT *
FROM bronze.orders
WHERE order_id IN (
    SELECT order_id
    FROM bronze.orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
OR order_id IS NULL
ORDER BY order_id;

In [0]:
%sql
CREATE OR REPLACE TABLE silver.orders AS
SELECT
    order_id,
    customer_id,
    restaurant_id,
    agent_id,
    delivery_location_id,
    order_status,
    payment_method,
    CAST(timestamp AS TIMESTAMP) AS order_timestamp,
    tip,
    total_amount
FROM bronze.orders
WHERE order_id IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE silver.restaurants AS
SELECT *
FROM bronze.restaurants;

CREATE OR REPLACE TABLE silver.locations AS
SELECT *
FROM bronze.locations;

CREATE OR REPLACE TABLE silver.menu_items AS
SELECT *
FROM bronze.menu_items;

CREATE OR REPLACE TABLE silver.order_items AS
SELECT
    order_id,
    item_id,
    item_name,
    CAST(item_price AS DECIMAL(10,2)) AS item_price,
    CAST(quantity AS INT) AS quantity
FROM bronze.order_items;